In [1]:
# HypotheSAEs Quickstart
# This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset
import os

%load_ext autoreload
%autoreload 2

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Set to '0' to use the first GPU, or 'cpu' to run on CPU
os.environ['OPENAI_KEY_SAE'] = 'EMPTY' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_
import numpy as np
import pandas as pd

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

/home/sevan/anaconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [2]:
CACHE_SIGNAL = "readmission"

In [5]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
val_ratio = 0.1  # Ratio of training data to use for validation
import sklearn
import sklearn.model_selection
few_shot_examples = 5
# base_dir = os.path.join(prefix, "demo-data")
# train_df = pd.read_json(os.path.join(base_dir, "yelp-demo-train-20K.json"), lines=True)
# val_df = pd.read_json(os.path.join(base_dir, "yelp-demo-val-2K.json"), lines=True)

# texts = train_df['text'].tolist()
# labels = train_df['stars'].values
# val_texts = val_df['text'].tolist() # These are only used for early stopping of SAE training, so we don't need labels.
from utils import df_to_prompts
import json
import re
base_dir = os.path.join(prefix, 'readmission_data')


# number_dict = {label: i for i, label in enumerate(label_type)}

# train_texts = df_to_prompts(few_shot_row, few_shot_label, train_X.iloc[few_shot_examples:, :], few_shot_examples=few_shot_examples)
pattern = re.compile(r"<think>.*?</think>", re.DOTALL)  # 匹配 <think> 到 </think>，包括换行
train_texts_idx_profile = []
test_texts_idx_profile = []
train_jsonl = os.path.join(base_dir, f"train.jsonl")
test_josnl = os.path.join(base_dir, f"test.jsonl")
with open(train_jsonl) as f:
    for line in f:
        obj = json.loads(line)
        profile = obj["profile"]
        profile = re.sub(pattern, "", profile)  # 删除 <think>...</think>
        obj["profile"] = profile.strip()
        train_texts_idx_profile.append((obj['profile'], obj['idx'], obj['label']))
    sorted_train_texts_idx_profile = sorted(train_texts_idx_profile, key=lambda x: x[1])
train_texts = [item[0] for item in sorted_train_texts_idx_profile]
train_y = [item[2] for item in sorted_train_texts_idx_profile]
with open(test_josnl)as f:
    for line in f:
        obj = json.loads(line)
        profile = obj["profile"]
        profile = re.sub(pattern, "", profile)  # 删除 <think>...</think>
        obj["profile"] = profile.strip()
        test_texts_idx_profile.append((obj['profile'], obj['idx'], obj['label']))
    sorted_test_texts_idx_profile = sorted(test_texts_idx_profile, key=lambda x: x[1])
test_texts = [item[0] for item in sorted_test_texts_idx_profile]
test_y = [item[2] for item in sorted_test_texts_idx_profile]
print(set(train_y), set(test_y))    
label_type = set(train_y).union(set(test_y))
number_dict = {label: 1 if label == 1 else 0 for label in label_type}
label_train = [number_dict[label] for label in train_y]
label_test = [number_dict[label] for label in test_y]
number_dict

{0, 1} {0, 1}


{0: 0, 1: 1}

In [6]:
#count label distribution
train_label_distribution = pd.Series(train_y).value_counts()
test_label_distribution = pd.Series(test_y).value_counts()
train_label_distribution,  test_label_distribution

(1    4758
 0    3328
 Name: count, dtype: int64,
 1    547
 0    352
 Name: count, dtype: int64)

In [7]:
texts, val_texts, labels, val_labels = sklearn.model_selection.train_test_split(
    train_texts, label_train, test_size=val_ratio, random_state=42, shuffle=True
)


In [8]:
len(test_y)

899

**Compute text embeddings for your dataset**

We'll compute text embeddings for a training set, and optionally a validation set. The validation embeddings are used for SAE eval and early-stopping during training.

Embeddings will be stored in the `emb_cache` directory (or `os.environ["EMB_CACHE_DIR"]` if you set it) using the `cache_name` parameter, so you only need to compute embeddings once.

You can use OpenAI or a local model.

Local models will run much faster on GPU. The default local model is `nomic-ai/modernbert-embed-base`. You can use any sentence-transformers model, but please read the model's docs; you may need to edit `get_local_embeddings`.

In [9]:
EMBEDDER = "Qwen/Qwen3-Embedding-0.6B" # OpenAI
# EMBEDDER = "nomic-ai/modernbert-embed-base" # Huggingface model, will run locally
CACHE_NAME = f"yelp_quickstart_{EMBEDDER}"

# text2embedding = get_openai_embeddings(texts + val_texts, model=EMBEDDER, cache_name=CACHE_NAME)
text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=32, cache_name=CACHE_NAME)
embeddings = np.stack([text2embedding[text] for text in texts])

train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])
text2embedding_test = get_local_embeddings(test_texts, model=EMBEDDER, batch_size=8, cache_name=CACHE_NAME)
test_embeddings = np.stack([text2embedding_test[text] for text in test_texts])

Loading embedding chunks: 100%|██████████| 10/10 [00:00<00:00, 47.36it/s]


Loaded 23906 embeddings in 0.2s
Loaded model Qwen/Qwen3-Embedding-0.6B to cuda


Processing chunks: 100%|██████████| 1/1 [01:43<00:00, 103.49s/it]


Saved 8086 embeddings to /home/sevan/myHypotheSAEs/emb_cache/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/chunk_010.npy


Loading embedding chunks: 100%|██████████| 11/11 [00:00<00:00, 31.10it/s]


Loaded 31992 embeddings in 0.4s
Loaded model Qwen/Qwen3-Embedding-0.6B to cuda


Processing chunks: 100%|██████████| 1/1 [00:12<00:00, 12.11s/it]


Saved 899 embeddings to /home/sevan/myHypotheSAEs/emb_cache/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/chunk_011.npy


**Train SAE(s)** 

Using different values of $M$ and $k$ will produce features at different levels of granularity. You can train multiple SAEs if you'd like to produce features at varying granularity, but this is optional.

See the README for more details about selecting $M$ and $k$.

In [10]:
checkpoint_dir = os.path.join(prefix, f"checkpoints_{CACHE_SIGNAL}", CACHE_NAME)
# sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
# sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
# sae_512_8 = train_sae(embeddings=train_embeddings, M=512, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_64 = train_sae(embeddings=train_embeddings, M=128, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_16 = train_sae(embeddings=train_embeddings, M=128, K=16, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_16 = train_sae(embeddings=train_embeddings, M=256, K=16, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_32 = train_sae(embeddings=train_embeddings, M=128, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_32 = train_sae(embeddings=train_embeddings, M=256, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_512_32 = train_sae(embeddings=train_embeddings, M=512, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_64 = train_sae(embeddings=train_embeddings, M=128, K=64, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_64 = train_sae(embeddings=train_embeddings, M=256, K=64, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_list = [sae_256_32, sae_128_64, sae_256_64]

 30%|██▉       | 295/1000 [00:21<00:50, 13.89it/s, train_loss=0.3998, val_loss=0.4383, dead_ratio=0.000]


Early stopping triggered after 296 epochs
Saved model to ./checkpoints_readmission/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=8.pt


 50%|████▉     | 498/1000 [00:35<00:35, 14.11it/s, train_loss=0.3025, val_loss=0.3275, dead_ratio=0.000]


Early stopping triggered after 499 epochs
Saved model to ./checkpoints_readmission/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=16.pt


 27%|██▋       | 274/1000 [00:20<00:53, 13.63it/s, train_loss=0.2759, val_loss=0.3377, dead_ratio=0.000]


...

Early stopping triggered after 436 epochs
Saved model to ./checkpoints_readmission/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=512_K=32.pt


100%|██████████| 1000/1000 [01:09<00:00, 14.33it/s, train_loss=0.2031, val_loss=0.2154, dead_ratio=0.000]


Saved model to ./checkpoints_readmission/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=64.pt


100%|██████████| 1000/1000 [01:11<00:00, 13.91it/s, train_loss=0.1500, val_loss=0.1762, dead_ratio=0.000]

Saved model to ./checkpoints_readmission/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=256_K=64.pt


**Interpret neurons**  

Interpret a random subset of neurons in the SAE to sanity-check that the learned features, and their interpretations, seem reasonable. We generate and print labels for `n_random_neurons` neurons, and we also print out the top-activating texts for each neuron.

In [11]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to Yelp, but you can customize this for your task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """You are a medical assistant. You need add hypothesis for each neuron, which should describe a specific aspect of the text.
Features should describe a specific aspect of patient profile. For example:
- "mentions patient has a tumor in the left lung"
- "indicates patient is allergic to penicillin"
""" 
# """All of the texts are reviews of restaurants on Yelp.
# Features should describe a specific aspect of the review. For example:
# - "mentions long wait times to receive service"
# - "praises how a dish was cooked, with phrases like 'perfect medium-rare'\""""
# Interpret random neurons
results = interpret_sae(
    texts=texts,
    embeddings=train_embeddings,
    sae=sae_list,
    n_random_neurons=30,
    print_examples_n=10,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model="Qwen/Qwen3-32B",
    annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
    max_interpretation_tokens= 1024,
)

Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 123.80it/s]


Activations shape: (7277, 640)


Generating 1 interpretation(s) per neuron: 100%|██████████| 30/30 [00:08<00:00,  3.48it/s]


Neuron 404 (from SAE M=256, K=64): <think>

</think>

- "mentions systemic lupus erythematosus (M32 or M35 or M06 or M48 or M33)

Top activating examples:
1. The patient is a 37-year-old male of Other/Unknown ethnicity with private insurance, presenting with a complex clinical profile and a hospital length of stay of 7.75 days, indicating a moderately prolonged inpatient admission. His current health status suggests the presence of several co-occurring psychiatric and medical conditions, including major depressive disorder (F30), adjustment disorder with mixed anxiety and depressed mood (F43), and alcohol use disorder (F10), which may contribute to both his psychological distress and functional impairment. Additionally, he has essential (primary) hypertension (I10) and unspecified systemic lupus erythematosus (M32), the latter of which may have contributed to his acute presentation and extended hospitalization. His BMI and blood pressure were not explicitly documented, but the presenc

**Generate hypotheses**

Generate hypotheses which are predictive of the target variable.

The `selection_method` parameter defines how we compute neuron predictiveness (see `src/select_neurons.py` for more details):
- "separation_score": E[target | top-activating examples] - E[target | zero-activating examples]
- "correlation": pearson(neuron activations, target variable)
- "lasso": select N nonzero features with an L1 regularized model

This cell outputs a dataframe with the following columns:
- `neuron_idx`: The index of the neuron in the SAE (if you're using multiple SAEs, this will be a global index across all of them).
- `source_sae`: The SAE that the neuron was selected from.
- `target_{selection_method}`: The predictiveness of the neuron for the target variable, using the selected `selection_method`.
- `interpretation`: The natural language interpretation of the neuron.
- `interp_fidelity_score`: The F1 fidelity score for how well the neuron's interpretation actually corresponds to its activation pattern.

In [12]:
RESULT_EXTRA_INFO = "20_Hypotheses"

In [13]:
selection_method = "lasso"
multi_class = True  # Set to True for direct multi-class classification, False for using one vs. rest classification
if len(set(labels)) == 2:
    selection_method = "lasso"  # Use logistic regression for binary classification
    multi_class = True
result_dir = f'result_cache/{CACHE_SIGNAL}_one_vs_rest_{RESULT_EXTRA_INFO}' if not multi_class else f'result_cache/{CACHE_SIGNAL}_multi_class_{RESULT_EXTRA_INFO}'

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
if multi_class:
    results = generate_hypotheses(
        texts=texts,
        labels=labels,
        embeddings=embeddings,
        sae=sae_list,
        cache_name=CACHE_NAME,
        selection_method=selection_method,
        n_selected_neurons=20,
        n_candidate_interpretations=20,
        task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
        classification=True,
        interpreter_model="Qwen/Qwen3-32B",
        annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
    )

    print("\nMost predictive features of Yelp reviews:")
    pd.set_option('display.max_colwidth', None)
    display(results.sort_values(by=f"target_{selection_method}", ascending=False))
    pd.reset_option('display.max_colwidth')
    results.to_csv(os.path.join(result_dir, f"hypotheses_{selection_method}.csv"), index=False)
else:
    results = {}
    for key in number_dict.keys():
        converted_key = {number_dict[dic_key]: 1 if dic_key == key else 0 for dic_key in number_dict.keys()}
        converted_label = [converted_key[label] for label in labels]
        print(len(converted_label))
        results[f'{key}_vs_rest'] = generate_hypotheses(
            texts=texts,
            labels=converted_label,
            embeddings=embeddings,
            sae=sae_list,
            cache_name=CACHE_NAME,
            selection_method=selection_method,
            n_selected_neurons=15,
            n_candidate_interpretations=3,
            task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
            classification=True,
            n_scoring_examples= 1000,  # Number of examples to use for scoring hypotheses
            interpreter_model="Qwen/Qwen3-32B",
            annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
        )
    for key, value in results.items():
        key_norm = key.replace("/", "or")
        key_norm = key_norm.replace(" ", "_")
        print(f"\nMost predictive features of Yelp reviews for {key}:")
        pd.set_option('display.max_colwidth', None)
        display(value.sort_values(by=f"target_{selection_method}", ascending=False))
        pd.reset_option('display.max_colwidth')
        results[key].to_csv(os.path.join(result_dir, f"hypotheses_{key_norm}_{selection_method}.csv"), index=False)
        

Embeddings shape: (7277, 1024)


Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 173.45it/s]

Activations shape: (7277, 640)

Step 1: Selecting top 20 predictive neurons
LASSO iteration   L1 Alpha # Features   Time (s)
----------------------------------------


       0   1.00e-01        635      21.20
       1   3.16e+01        180       0.79
       2   5.62e+02          0       0.11
       3   1.33e+02         24       0.22
       4   2.74e+02          7       0.16
       5   1.91e+02         11       0.20
       6   1.60e+02         14       0.20
       7   1.46e+02         20       0.20

Found alpha=1.46e+02 yielding exactly 20 features
Total search time: 23.08s

Step 2: Interpreting selected neurons


Generating 20 interpretation(s) per neuron: 100%|██████████| 400/400 [01:08<00:00,  5.85it/s]


...

Scoring neuron interpretation fidelity (20 neurons; 20 candidate interps per neuron; 100 examples to score each interp):  47%|████▋     | 18690/40000 [12:10<16:22, 21.68it/s] 

API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Request timed out.; retrying in 10.0s... (2/3)
API error: Req

Scoring neuron interpretation fidelity (20 neurons; 20 candidate interps per neuron; 100 examples to score each interp): 100%|██████████| 40000/40000 [26:34<00:00, 25.08it/s]   



Most predictive features of Yelp reviews:


,neuron_idx,source_sae,target_lasso,interpretation,f1_fidelity_score
0,135,"(256, 32)",0.142452,"<think>\n\n</think>\n\n- ""indicates in-hospital mortality",0.901099
1,372,"(128, 64)",0.124129,"<think>\n\n</think>\n\n- ""mentions multiple trauma-related injuries",0.979592
2,142,"(256, 32)",0.076623,"<think>\n\n</think>\n\n- ""mentions a diagnosis of type 2 diabetes mellitus (E11 or E05)",0.938298
3,294,"(128, 64)",0.057431,"<think>\n\n</think>\n\n- ""mentions a primary diagnosis related to a non-life-threatening or unspecified condition without significant chronic comorbidities",0.863636
4,345,"(128, 64)",0.049215,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay (14 days or more) without inpatient mortality",0.708732
5,555,"(256, 64)",0.046716,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or related renal conditions such as acute kidney injury, nephrotic syndrome, or unspecified kidney disease",1.000000
6,456,"(256, 64)",0.040457,"<think>\n\n</think>\n\n- ""mentions traumatic injuries involving specific body regions with ICD-10 codes starting with S (e.g., S01, S02, S22, S37, S52, S62",0.969072
7,364,"(128, 64)",0.040450,"<think>\n\n</think>\n\n- ""mentions sepsis as a primary or contributing diagnosis",1.000000
8,362,"(128, 64)",0.040323,"<think>\n\n</think>\n\n- ""mentions in-hospital mortality (MORT = 1) or deceased status as part of the clinical outcome",0.734177
9,109,"(256, 32)",0.029714,"<think>\n\n</think>\n\n- ""indicates patient has chronic liver disease or cirrhosis",0.936170


**Save all in one json**

In [14]:
import os
import json
if not os.path.isdir(result_dir):
    os.makedirs(result_dir)
train_all_info_file = os.path.join(result_dir, "train_all_info.jsonl")
test_all_info_file = os.path.join(result_dir, "test_all_info.jsonl")
with open(test_all_info_file, 'w') as f:
    for text, label, idx in zip(test_texts, test_y, range(len(test_texts))):
        f.write(json.dumps({"idx": idx, "profile": text, "label": label}) + "\n")
with open(train_all_info_file, 'w') as f:
    for text, label, idx in zip(texts, labels, range(len(texts))):
        f.write(json.dumps({"idx": idx, "profile": text, "label": label}) + "\n")

In [15]:
from hypothesaes.sae import SparseAutoencoder, load_model, get_multiple_sae_activations, get_sae_checkpoint_name
activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, embeddings, return_neuron_source_info=True)
test_activations, neuron_source_sae_info_test = get_multiple_sae_activations(sae_list, test_embeddings, return_neuron_source_info=True)

Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 577.01it/s]


In [16]:
def clean_interpretation(text):
    return text.replace("<think>", "").replace("</think>", "").strip().lstrip("-").strip().strip('"')

# 创建解释字典
if multi_class:
    neuron_interpretations = {
        row['neuron_idx']: clean_interpretation(row['interpretation'])
        for _, row in results.iterrows()
    }
else:
    neuron_interpretations = {}
    for key, value in results.items():
        neuron_interpretations[key] = {
            row['neuron_idx']: clean_interpretation(row['interpretation'])
            for _, row in value.iterrows()
        }

In [17]:
import torch
def describe_from_activations(activations: np.array, neuron_interpretations: dict, threshold=0.5, extra_info =None):
    """根据 activation 向量和解释字典，返回解释列表"""

    
    descriptions = []
    for idx, value in enumerate(activations):
        if value.item() > threshold and idx in neuron_interpretations:
            if extra_info is not None:
                descriptions.append((idx, neuron_interpretations[idx], value.item(), extra_info))
            else:
                descriptions.append((idx, neuron_interpretations[idx], value.item()))
    
    # 可以按激活值排序
    descriptions = sorted(descriptions, key=lambda x: -x[2])
    return descriptions
def render_descriptions(descriptions, multiclass=False):
    if not descriptions:
        return "No meaningful activations found."

    output = ["This text likely involves:"]
    if multiclass:   
        for idx, interp, score in descriptions:
            output.append(f"- {interp} (activation={score:.2f})")
    else:
        for idx, interp, score, type in descriptions:
            output.append(f"- {interp} (activation={score:.2f}, type={type})")
    return "\n".join(output)

In [18]:
interpret_list = []
if multi_class:
    for act in activations:
        descriptions = describe_from_activations(act, neuron_interpretations, threshold=0)
        interpret_list.append(render_descriptions(descriptions, True))
else:
    for act in activations:
        descriptions = []
        for key, value in neuron_interpretations.items():
            descriptions.extend(describe_from_activations(act, value, threshold=0, extra_info=f'[{key}]'))
        interpret_list.append(render_descriptions(descriptions, False))
with open(os.path.join(result_dir, 'test_interpretations.jsonl'), "w") as f:
    for text, interpret in zip(range(len(interpret_list)), interpret_list):
        f.write(json.dumps({"text": text, "interpretation": interpret}) + "\n")
interpret_list_test = []
if multi_class:
    for act in test_activations:
        descriptions = describe_from_activations(act, neuron_interpretations, threshold=0)
        interpret_list_test.append(render_descriptions(descriptions, True))
else:
    for act in test_activations:
        descriptions = []
        for key, value in neuron_interpretations.items():
            descriptions.extend(describe_from_activations(act, value, threshold=0, extra_info=f'[{key}]'))
        interpret_list_test.append(render_descriptions(descriptions, False))
with open(os.path.join(result_dir, 'test_interpretations.jsonl'), "w") as f:
    for text, interpret in zip(range(len(interpret_list_test)), interpret_list_test):
        f.write(json.dumps({"text": text, "interpretation": interpret}) + "\n")
    

**Evaluate held-out generalization**

Finally, we evaluate whether these are good hypotheses by testing whether their natural language interpretations can predict the target variable.  

We compute annotations for each hypothesized concept on a holdout set (not seen during SAE training & feature selection).

After annotation, we output a dataframe with the following columns:
- `hypothesis`: The natural language hypothesis (which came from interpreting a predictive neuron in the SAE)
- `separation_score`: How much the target variable differs when the concept is present vs. absent (i.e., $E[Y\mid\text{concept} = 1] - E[Y\mid\text{concept} = 0]$).
- `separation_pvalue`: The t-test p-value of the null hypothesis that the separation score is 0 (i.e., the concept is not associated with the target variable).
- `regression_coef`: The coefficient of the concept in a multivariate linear regression of the target variable on all concepts.
- `regression_pval`: The p-value of the null hypothesis that the regression coefficient is 0.
- `feature_prevalence`: The fraction of examples that contain the concept.

Additionally, we output the evaluation metrics used in the paper:
- Significant hypotheses: the number of hypotheses that are significant in the multivariate regression at a specified significance level (default $0.1$) after Bonferroni correction. You can pass in a different significance level using the `corrected_pval_threshold` parameter.
- AUC or $R^2$: how well the hypotheses collectively predict the target variable in the multivariate regression.


In [19]:
# holdout_df = pd.read_json(os.path.join(base_dir, "yelp-demo-holdout-2K.json"), lines=True)
# holdout_texts = holdout_df['text'].tolist()
# holdout_labels = holdout_df['stars'].values
from hypothesaes.quickstart import evaluate_hypotheses
if multi_class:
    metrics, evaluation_df = evaluate_hypotheses(
        hypotheses_df=results,
        texts=test_texts,
        labels=label_test,
        cache_name=CACHE_NAME,
        max_words_per_example=1024,
        annotator_model="Qwen/Qwen3-32B",  # Use the same model as for training
        classification=True
    )

    pd.set_option('display.max_colwidth', None)
    display(evaluation_df)
    pd.reset_option('display.max_colwidth')

    print("\nHoldout Set Metrics:")
    print(f"R² Score: {metrics['r2']:.3f}")
    print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
        f"(p < {metrics['Significant'][2]:.3e})")
else:
    metrics, evaluation_df = {}, {}
    for key in number_dict.keys():
        key_norm = key.replace("/", "or")
        key_norm = key_norm.replace(" ", "_")
        converted_key = {number_dict[dic_key]: 1 if dic_key == key else 0 for dic_key in number_dict.keys()}
        converted_label = [converted_key[label] for label in label_test]
        metrics[key], evaluation_df[key] = evaluate_hypotheses(
            hypotheses_df=results[f'{key}_vs_rest'],
            texts=test_texts,
            labels=converted_label,
            cache_name=os.path.join(CACHE_NAME, CACHE_SIGNAL, f"{key}_vs_rest"),
            max_words_per_example=1024,
            annotator_model="Qwen/Qwen3-32B",  # Use the same model as for training
            classification=True
        )
        evaluation_df[key].to_csv(os.path.join(result_dir, f"evaluation_{key_norm}.csv"), index=False)
        with open(os.path.join(result_dir, f"metrics_{key_norm}.json"), "w") as f:
            json.dump(metrics[key], f, indent=4)
        pd.set_option('display.max_colwidth', None)
        display(evaluation_df[key])
        pd.reset_option('display.max_colwidth')
        print(f"\nHoldout Set Metrics for {key}:")
        print(f"R² Score: {metrics[key]['r2']:.3f}")
        print(f"Significant hypotheses: {metrics[key]['Significant'][0]}/{metrics[key]['Significant'][1]} " 
            f"(p < {metrics[key]['Significant'][2]:.3e})")


Step 1: Annotating texts with 20 hypotheses
Found 0 cached items; annotating 17980 uncached items


Annotating: 100%|██████████| 17980/17980 [12:30<00:00, 23.95it/s]


Step 2: Computing predictiveness of hypothesis annotations
Optimization terminated successfully.
         Current function value: 0.627858
         Iterations 6


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
17,"<think>\n\n</think>\n\n- ""mentions type 2 diabetes mellitus (E11) as a primary or contributing diagnosis",0.165132,0.000004,0.770819,0.267081,0.284761
2,"<think>\n\n</think>\n\n- ""mentions a diagnosis of type 2 diabetes mellitus (E11 or E05)",0.155644,0.000012,-0.128866,0.851202,0.295884
5,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or related renal conditions such as acute kidney injury, nephrotic syndrome, or unspecified kidney disease",0.081965,0.016847,0.197559,0.729835,0.342603
10,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or acute kidney injury as a primary or contributing diagnosis",0.079608,0.021888,0.001024,0.998611,0.325918
13,"<think>\n\n</think>\n\n- ""mentions anemia-related diagnoses (e.g., anemia due to chronic disease, iron deficiency anemia, unspecified anemia)",0.077907,0.028257,0.178679,0.274722,0.300334
19,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay exceeding 18 days",0.052305,0.111871,0.465463,0.054592,0.430478
9,"<think>\n\n</think>\n\n- ""indicates patient has chronic liver disease or cirrhosis",0.040692,0.479692,0.238545,0.376005,0.087875
18,"<think>\n\n</think>\n\n- ""mentions multiple nonspecific or psychiatric symptoms with no clear organic diagnosis",0.030559,0.472213,0.177538,0.363039,0.179088
12,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay (typically exceeding 4 days) due to multiple traumatic injuries or complex clinical conditions requiring extended inpatient care",0.027829,0.406343,0.124123,0.492001,0.616240
16,"<think>\n\n</think>\n\n- ""mentions a diagnosis of migraine (G40, G43, G45, G80, or G89)",0.023166,0.754178,0.146028,0.653187,0.051168



Holdout Set Metrics:
R² Score: 0.062
Significant hypotheses: 1/20 (p < 5.000e-03)


In [20]:
if multi_class:
        pd.set_option('display.max_colwidth', None)
        display(evaluation_df)
        pd.reset_option('display.max_colwidth')
        print("\nHoldout Set Metrics:")
        print(f"R² Score: {metrics['r2']:.3f}")
        print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
            f"(p < {metrics['Significant'][2]:.3e})")
        evaluation_df.to_csv(os.path.join(result_dir, "evaluation.csv"), index=False)
        with open(os.path.join(result_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=4)
        
else:
        for key in metrics.keys():
                # print(metrics[key])
                pd.set_option('display.max_colwidth', None)
                display(evaluation_df[key])
                pd.reset_option('display.max_colwidth')
                print(f"\nHoldout Set Metrics for {key}:")
                print(f"R² Score: {metrics[key]['r2']:.3f}")
                print(f"Significant hypotheses: {metrics[key]['Significant'][0]}/{metrics[key]['Significant'][1]} " 
                f"(p < {metrics[key]['Significant'][2]:.3e})")
                evaluation_df[key].to_csv(os.path.join(result_dir, f"evaluation_{key}.csv"), index=False)
                with open(os.path.join(result_dir, f"metrics_{key}.json"), "w") as f:
                    json.dump(metrics[key], f, indent=4)

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
17,"<think>\n\n</think>\n\n- ""mentions type 2 diabetes mellitus (E11) as a primary or contributing diagnosis",0.165132,0.000004,0.770819,0.267081,0.284761
2,"<think>\n\n</think>\n\n- ""mentions a diagnosis of type 2 diabetes mellitus (E11 or E05)",0.155644,0.000012,-0.128866,0.851202,0.295884
5,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or related renal conditions such as acute kidney injury, nephrotic syndrome, or unspecified kidney disease",0.081965,0.016847,0.197559,0.729835,0.342603
10,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or acute kidney injury as a primary or contributing diagnosis",0.079608,0.021888,0.001024,0.998611,0.325918
13,"<think>\n\n</think>\n\n- ""mentions anemia-related diagnoses (e.g., anemia due to chronic disease, iron deficiency anemia, unspecified anemia)",0.077907,0.028257,0.178679,0.274722,0.300334
19,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay exceeding 18 days",0.052305,0.111871,0.465463,0.054592,0.430478
9,"<think>\n\n</think>\n\n- ""indicates patient has chronic liver disease or cirrhosis",0.040692,0.479692,0.238545,0.376005,0.087875
18,"<think>\n\n</think>\n\n- ""mentions multiple nonspecific or psychiatric symptoms with no clear organic diagnosis",0.030559,0.472213,0.177538,0.363039,0.179088
12,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay (typically exceeding 4 days) due to multiple traumatic injuries or complex clinical conditions requiring extended inpatient care",0.027829,0.406343,0.124123,0.492001,0.616240
16,"<think>\n\n</think>\n\n- ""mentions a diagnosis of migraine (G40, G43, G45, G80, or G89)",0.023166,0.754178,0.146028,0.653187,0.051168



Holdout Set Metrics:
R² Score: 0.062
Significant hypotheses: 1/20 (p < 5.000e-03)


In [21]:
pd.set_option('display.max_colwidth', None)
display(evaluation_df)
pd.reset_option('display.max_colwidth')
print("\nHoldout Set Metrics:")
# print(f"R² Score: {metrics['r2']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
    f"(p < {metrics['Significant'][2]:.3e})")
evaluation_df.to_csv(os.path.join(result_dir, "evaluation.csv"), index=False)
with open(os.path.join(result_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
17,"<think>\n\n</think>\n\n- ""mentions type 2 diabetes mellitus (E11) as a primary or contributing diagnosis",0.165132,0.000004,0.770819,0.267081,0.284761
2,"<think>\n\n</think>\n\n- ""mentions a diagnosis of type 2 diabetes mellitus (E11 or E05)",0.155644,0.000012,-0.128866,0.851202,0.295884
5,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or related renal conditions such as acute kidney injury, nephrotic syndrome, or unspecified kidney disease",0.081965,0.016847,0.197559,0.729835,0.342603
10,"<think>\n\n</think>\n\n- ""mentions chronic kidney disease or acute kidney injury as a primary or contributing diagnosis",0.079608,0.021888,0.001024,0.998611,0.325918
13,"<think>\n\n</think>\n\n- ""mentions anemia-related diagnoses (e.g., anemia due to chronic disease, iron deficiency anemia, unspecified anemia)",0.077907,0.028257,0.178679,0.274722,0.300334
19,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay exceeding 18 days",0.052305,0.111871,0.465463,0.054592,0.430478
9,"<think>\n\n</think>\n\n- ""indicates patient has chronic liver disease or cirrhosis",0.040692,0.479692,0.238545,0.376005,0.087875
18,"<think>\n\n</think>\n\n- ""mentions multiple nonspecific or psychiatric symptoms with no clear organic diagnosis",0.030559,0.472213,0.177538,0.363039,0.179088
12,"<think>\n\n</think>\n\n- ""mentions a prolonged hospital length of stay (typically exceeding 4 days) due to multiple traumatic injuries or complex clinical conditions requiring extended inpatient care",0.027829,0.406343,0.124123,0.492001,0.616240
16,"<think>\n\n</think>\n\n- ""mentions a diagnosis of migraine (G40, G43, G45, G80, or G89)",0.023166,0.754178,0.146028,0.653187,0.051168



Holdout Set Metrics:
Significant hypotheses: 1/20 (p < 5.000e-03)


In [22]:
evaluation_df
with open(os.path.join(result_dir, 'hypotheses_info.jsonl'), "w") as f:
    for _, row in evaluation_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

In [23]:
#output hypothesis and its coef in csv
df_coef = evaluation_df[['hypothesis', 'regression_coef']]
df_coef.to_csv(os.path.join(result_dir, 'hypothesis_coef.csv'), index=False)